In [23]:
# Convert the hypergraph.csv to .pt file
import pandas as pd

# 1. calculate the number of hyperedges
df = pd.read_csv('hypergraph_construct/hypergraph_index.csv')
num_rows = len(df)
data_dict = {'N_edges': num_rows}
print(data_dict)

{'N_edges': 33548}


In [24]:
# 2. calculate the number of nodes
unique_drugs = set()
df['drug'].apply(lambda x: unique_drugs.update(eval(x)))
num_unique_drugs = len(unique_drugs)

unique_diseases = df['disease'].unique()
num_unique_diseases = len(unique_diseases)
data_dict['N_nodes'] = num_unique_drugs + num_unique_diseases
data_dict['num_drugs']=num_unique_drugs
data_dict['num_diseases']=num_unique_diseases
print(data_dict)
print(num_unique_drugs)
print(num_unique_diseases)

{'N_edges': 33548, 'N_nodes': 5575, 'num_drugs': 2774, 'num_diseases': 2801}
2774
2801


In [25]:
# 3. obtain NodeEdgePair and EdgeNodePair
# 3.1 assign hypergraph_indices to nodes
node_index = {}
index = 0

for node in unique_drugs:
    node_index[node] = index
    index += 1

for node in unique_diseases:
    node_index[node] = index
    index += 1

print(node_index) 
print(len(node_index))



{16409: 0, 16410: 1, 16419: 2, 16430: 3, 16434: 4, 16443: 5, 16450: 6, 16459: 7, 16465: 8, 16466: 9, 16468: 10, 16469: 11, 16471: 12, 16473: 13, 16475: 14, 16477: 15, 16478: 16, 16480: 17, 16490: 18, 16495: 19, 16496: 20, 16502: 21, 16505: 22, 16506: 23, 16508: 24, 16510: 25, 16511: 26, 16512: 27, 16513: 28, 16517: 29, 16519: 30, 16521: 31, 16523: 32, 16531: 33, 16532: 34, 16538: 35, 16544: 36, 16545: 37, 16550: 38, 16554: 39, 16557: 40, 16558: 41, 16559: 42, 16570: 43, 16572: 44, 16573: 45, 16574: 46, 16575: 47, 16578: 48, 16579: 49, 16580: 50, 16581: 51, 16582: 52, 16587: 53, 16588: 54, 16589: 55, 16590: 56, 16592: 57, 16593: 58, 16595: 59, 16631: 60, 16634: 61, 16635: 62, 16640: 63, 16667: 64, 16671: 65, 16672: 66, 16674: 67, 16675: 68, 16677: 69, 16687: 70, 16688: 71, 16689: 72, 16690: 73, 16691: 74, 16693: 75, 16694: 76, 16697: 77, 16698: 78, 16699: 79, 16711: 80, 16714: 81, 16719: 82, 16720: 83, 16721: 84, 16723: 85, 16740: 86, 16741: 87, 16744: 88, 16753: 89, 16758: 90, 16783: 9

In [26]:
# 3.2 assign hypergraph_indices to hyperedges
edge_index = {}
for idx, row in df.iterrows():
    drugs = eval(row['drug'])
    disease = row['disease']
    combined = tuple(drugs + [disease])
    edge_index[combined] = idx
#print(edge_index)
print(len(edge_index))  

33548


In [27]:
# 3.3 obtain NodeEdgePair and EdgeNodePair
NodeEdgePair = []
EdgeNodePair = []

for edge, edge_idx in edge_index.items():
    for node in edge:
        node_idx = node_index[node]
        NodeEdgePair.append([node_idx, edge_idx])
        EdgeNodePair.append([edge_idx, node_idx])

data_dict['NodeEdgePair'] = NodeEdgePair
data_dict['EdgeNodePair'] = EdgeNodePair

print("NodeEdgePair:", NodeEdgePair[:10])
print("EdgeNodePair:", EdgeNodePair[:10])

print(len(NodeEdgePair))
print(len(EdgeNodePair))

print(data_dict.keys())


NodeEdgePair: [[70, 0], [2774, 0], [70, 1], [2775, 1], [705, 2], [2774, 2], [705, 3], [2775, 3], [75, 4], [2774, 4]]
EdgeNodePair: [[0, 70], [0, 2774], [1, 70], [1, 2775], [2, 705], [2, 2774], [3, 705], [3, 2775], [4, 75], [4, 2774]]
103556
103556
dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair'])


In [28]:
# 4.obtain nodewt and edgewt
import torch

nodewt = [0] * len(node_index)
edgewt = [0] * len(edge_index)

for edge, edge_idx in edge_index.items():
    for node in edge:
        node_idx = node_index[node]
        nodewt[node_idx] += 1
        edgewt[edge_idx] += 1

nodewt_tensor = torch.tensor(nodewt, dtype=torch.float)
edgewt_tensor = torch.tensor(edgewt, dtype=torch.float)

print("Node Weights:", nodewt_tensor)
print("Edge Weights:", edgewt_tensor)

print(len(nodewt_tensor))
print(len(edgewt_tensor))

data_dict['nodewt'] = nodewt_tensor
data_dict['edgewt'] = edgewt_tensor

print(data_dict.keys())



Node Weights: tensor([12.,  1., 12.,  ...,  1.,  1.,  1.])
Edge Weights: tensor([2., 2., 2.,  ..., 3., 3., 3.])
5575
33548
dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt'])


In [29]:
# 5.add node_index_hypergraph and edge_index_hypergraph to data_dict
data_dict['node_index_hypergraph'] = node_index
data_dict['edge_index_hypergraph'] = edge_index
print(data_dict.keys())

dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt', 'node_index_hypergraph', 'edge_index_hypergraph'])


In [30]:
# 6. match the pretrained node features to the nodes in hypergraph
# 6.1 read the pretrained feature file
pretrain_feature_data = torch.load('data_split/pretrain_node_feature512.pt')
print(type(pretrain_feature_data))
drug_feature_tensor=pretrain_feature_data['drug']
disease_feature_tensor=pretrain_feature_data['disease']
print(drug_feature_tensor.shape)
print(disease_feature_tensor.shape)


<class 'dict'>
torch.Size([6681, 512])
torch.Size([17080, 512])


In [ ]:
# 6.2 initialize the feature matrix
kg_data = pd.read_csv('data_split/kg_directed.csv')

import numpy as np
num_nodes = len(node_index)
feature_dim = drug_feature_tensor.shape[1]  
node_feat = np.zeros((num_nodes, feature_dim))
print(node_feat.shape)

(5575, 512)


In [33]:
# 6.3 obtain the pretrained feature matrix
for node_name, row_index in node_index.items():  
    node_info = kg_data[(kg_data['x_index'] == node_name) | (kg_data['y_index'] == node_name)].iloc[0]
    if node_info['x_index'] == node_name:
        node_type = node_info['x_type']
        feature_idx = int(node_info['x_idx'])
    else:
        node_type = node_info['y_type']
        feature_idx = int(node_info['y_idx'])
        
    if row_index <= 2773 and node_type == 'drug':
        node_feat[row_index] = drug_feature_tensor[feature_idx]
    elif row_index > 2773 and node_type == 'disease':
        node_feat[row_index] = disease_feature_tensor[feature_idx]
print(node_feat.shape)
print(node_feat)



IndexError: single positional indexer is out-of-bounds

In [11]:
# 6.4 add the pretrained feature matrix to data_dict
data_dict['node_feat'] = node_feat
print(data_dict.keys())

dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt', 'node_index_hypergraph', 'edge_index_hypergraph', 'node_feat'])


In [21]:
""" # 6b. randomly initialize the feature matrix
import numpy as np
num_nodes = len(node_index)
feature_dim = 512
node_feat = np.random.rand(num_nodes, feature_dim)  

data_dict['node_feat'] = node_feat
print(node_feat.shape)
print(node_feat)
print(data_dict.keys())
torch.save(data_dict, 'data_split/hypergraph_random_feature.pt') """

(5575, 512)
[[0.95334128 0.01090344 0.10946074 ... 0.7527175  0.82958692 0.80755117]
 [0.51841827 0.43810097 0.87658622 ... 0.86959307 0.55190365 0.47018076]
 [0.48148454 0.72324227 0.00523449 ... 0.79114839 0.45278241 0.95901358]
 ...
 [0.14548165 0.33758234 0.31411301 ... 0.45475087 0.13351795 0.78387292]
 [0.95727461 0.33261405 0.47933768 ... 0.40312741 0.09738885 0.82303621]
 [0.6947069  0.42612337 0.12526887 ... 0.88521457 0.75079967 0.58869183]]
dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt', 'node_index_hypergraph', 'edge_index_hypergraph', 'node_feat'])


In [12]:
# 7. save the data_dict to pt file
torch.save(data_dict, 'data_split/hypergraph_pretrain_allKG.pt')


In [13]:
# 8. check the pt file
data = torch.load('data_split/hypergraph_pretrain_allKG.pt')
print(data.keys())



dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt', 'node_index_hypergraph', 'edge_index_hypergraph', 'node_feat'])
